In [1]:
import json
import operator
import re
from pathlib import Path
from typing import TypedDict, Annotated

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, START, END

llm      = ChatOllama(model="qwen2.5:latest", temperature=0)
DATA_DIR = Path("./data")

print(f"✅ Setup")


✅ Setup


In [ ]:
# Schema di stato condiviso tra tutti i nodi
class StatoRichiesta(TypedDict, total=False):
    # Input
    richiesta:              str     # messaggio originale del cliente
    claim_id:               str     # ID della pratica
    customer_policy_id:     str     # ID dell'assicurazione del cliente
    customer_loss_type:     str     # tipo di pratica dichiarata dall'utente (incidente, furto, allagamento)
    estimated_damage:       int     # valore stimato dell'indennizzo
    documents:              list    # lista dei documenti inviati dal cliente (come stringhe)

    # Risultati intermedi
    customer_name:          str     # prende il nome del cliente (probabilmente inutile, poi vediamo)
    risk_tier:              str     # possibile approfondimento futuro (sale con il numeri di incidenti)
    policy_id:              str     # prende l'assicurazione effettiva (bisogna verificare che sia effettivamente posseduta dal cliente)
    product:                str     # prodotto assicurato (casa / macchina)
    coverage:               str     # da cosa ti copre l'assicurazione (bisogna verificare anche questo)
    deductible:             int     # franchigia (parte che il cliente deve pagare comunque)
    max_claims:             int     # tetto massimo coperto (probabilmente oltre questa soglia il perito non può agisce)
    active:                 bool    # flag che indica se l'assicurazione è attivo o meno (fondamentale)

    # Approfondimento di loss_type
    loss_type:              str     # tipo di pratica
    required_documents:     list    # lista dei documenti richiesti dall'assicurazione (come stringhe)
    automatic_review_max:   int     # soglia massima per indennizzo automatico

    # INFO DI SISTEMA
    esito:                  str    # APPROVATO | ESCALATION | RIFIUTO | AMBIGUO
    motivazione:            str    # motivazione dell'esito
    risposta_finale:        str    # risposta finale data dall'agente
    retry_count:            int    # quanti cicli di chiarimento
    chiarimento:            str    # risposta del cliente in caso di ambiguità
    errori:                 list   # lista degli errori riscontrati (non so se tenerla)

print("Campi dello stato:", list(StatoRichiesta.__annotations__.keys()))

Campi dello stato: ['richiesta', 'claim_id', 'customer_policy_id', 'customer_loss_type', 'estimated_damage', 'documents', 'customer_name', 'risk_tier', 'policy_id', 'product', 'coverage', 'deductible', 'max_claims', 'active', 'loss_type', 'required_documents', 'automatic_review_max', 'esito', 'motivazione', 'risposta_finale', 'retry_count', 'chiarimento', 'errori']


In [ ]:
def leggi_json():
    pass

In [ ]:
def regole_policy():
    pass

In [ ]:
def claims_review_agent():
    pass

In [ ]:
builder = StateGraph(StatoRichiesta)

# Aggiungi nodi
builder.add_node("estrai_parametri",   estrai_parametri)
builder.add_node("verifica_budget",    verifica_budget)
builder.add_node("cerca_catalogo",     cerca_catalogo)
builder.add_node("decide",             decide)
builder.add_node("chiedi_chiarimento", chiedi_chiarimento)

# Archi fissi (sequenza lineare)
builder.add_edge(START,              "estrai_parametri")
builder.add_edge("estrai_parametri", "verifica_budget")
builder.add_edge("verifica_budget",  "cerca_catalogo")
builder.add_edge("cerca_catalogo",   "decide")

# Arco condizionale dopo decide (router)
builder.add_conditional_edges(
    "decide",
    router_dopo_decide,
    {END: END, "chiedi_chiarimento": "chiedi_chiarimento"}
)

# Ciclo: dopo il chiarimento si torna a decide (non si riesegue tutta la pipeline)
builder.add_edge("chiedi_chiarimento", "decide")

graph = builder.compile()
print("✅ Grafo compilato — nodi:", list(builder.nodes.keys()))